In [1]:
from rapidfuzz import process, fuzz
import pandas as pd
import numpy as np
from pathlib import Path
import re

In [2]:
path_dflb = Path(r"E:\ProyectoAnalisisElectrico\BarrasEstaciones\LocatedBars.csv") # df_located_bars
path_dfp = Path(r"E:\ProyectoAnalisisElectrico\DiaPromedio\Periodos\2505_2604\2505_2604_mean_period.parquet")
dflb = pd.read_csv(path_dflb, sep=";", encoding="utf-8")
dfp = pd.read_parquet(path_dfp).reset_index()

In [3]:
dflb.head()

,Nombre Subestación,ID,Nombre,Nombre Centro Control,Nombre Propietario,Nombre Coordinado,Número,Nemotecnico,Descripcion,ID_E,Número_E,Nemotecnico_E,Región,Provincia,Comuna,Macrozona
0,S/E CENTRAL ALFALFAL,1,BA S/E CENTRAL ALFALFAL 12KV BP1,AES ANDES S.A.,AES ANDES S.A.,AES ANDES S.A.,1,BA01G0010SE001G0010,NaN,199,1,SE001G0010,Metropolitana de Santiago,Cordillera,San José de Maipo,Centro
1,S/E CENTRAL ALFALFAL,2,BA S/E CENTRAL ALFALFAL 12KV BP2,AES ANDES S.A.,AES ANDES S.A.,AES ANDES S.A.,2,BA02G0010SE001G0010,NaN,199,1,SE001G0010,Metropolitana de Santiago,Cordillera,San José de Maipo,Centro
2,S/E CENTRAL MAITENES,6,BA S/E CENTRAL MAITENES 6.6KV B1,AES ANDES S.A.,AES ANDES S.A.,AES ANDES S.A.,1,BA01G0010SE004G0010,NaN,201,4,SE004G0010,Metropolitana de Santiago,Cordillera,San José de Maipo,Centro
3,S/E CENTRAL QUELTEHUES,7,BA S/E CENTRAL QUELTEHUES 110KV BP1,AES ANDES S.A.,AES ANDES S.A.,AES ANDES S.A.,1,BA01G0010SE006G0010,NaN,203,6,SE006G0010,Metropolitana de Santiago,Cordillera,San José de Maipo,Centro
4,S/E CENTRAL QUELTEHUES,8,BA S/E CENTRAL QUELTEHUES 12KV,AES ANDES S.A.,AES ANDES S.A.,AES ANDES S.A.,2,BA02G0010SE006G0010,NaN,203,6,SE006G0010,Metropolitana de Santiago,Cordillera,San José de Maipo,Centro


In [4]:
df = dflb[['Nombre',"Nombre Subestación", "Macrozona", "Región"]].copy()

In [5]:
df.head()

,Nombre,Nombre Subestación,Macrozona,Región
0,BA S/E CENTRAL ALFALFAL 12KV BP1,S/E CENTRAL ALFALFAL,Centro,Metropolitana de Santiago
1,BA S/E CENTRAL ALFALFAL 12KV BP2,S/E CENTRAL ALFALFAL,Centro,Metropolitana de Santiago
2,BA S/E CENTRAL MAITENES 6.6KV B1,S/E CENTRAL MAITENES,Centro,Metropolitana de Santiago
3,BA S/E CENTRAL QUELTEHUES 110KV BP1,S/E CENTRAL QUELTEHUES,Centro,Metropolitana de Santiago
4,BA S/E CENTRAL QUELTEHUES 12KV,S/E CENTRAL QUELTEHUES,Centro,Metropolitana de Santiago


vamos a buscar alguna forma de rescatar las palabras clave

In [6]:
df['NS1'] = df['Nombre Subestación'].str.replace(r'^S/E\s*', '', regex=True)
df['N1'] = df['Nombre'].str.replace(r'BA\s+S/E\s*', '', regex=True)

df['NS1'] = df['NS1'].str.replace(r'\(.*?\)', '', regex=True).str.strip()
df['N1'] = df['N1'].str.replace(r'\(.*?\)', '', regex=True).str.strip()

df['N1'] = df['N1'].str.replace(r'\d+(?:[.,]\d+)?\s*[kK][vV].*', '', regex=True).str.strip()

df['NS1'] = df['NS1'].str.replace(r'(?i)\bCENTRAL\b\s*', '', regex=True).str.strip()

df['NS1'] = df['NS1'].str.replace(r'(?i)TAP\s+OFF', 'T.OFF', regex=True)
# Eliminamos los números seguidos de KV en la columna NS1
df['NS1'] = df['NS1'].str.replace(r'(?i)\s*\d+\s*KV', '', regex=True).str.strip()
# Reemplazamos cualquier tipo de espacio (uno o varios) por nada (texto vacío)
df['NS1'] = df['NS1'].str.replace(r'\s+', '', regex=True)

In [35]:
df["Voltaje"] = df['Nombre'].str.extract(r'(\d+(?:[.,]\d+)?)\s*[kK][vV]')
df.loc[df['Nombre'].isin(['BA S/E MONTURAQUI 4.16 BP1', 'BA S/E MONTURAQUI 4.16 BP2']), 'Voltaje'] = "4.16"
df.loc[df['Nombre'] == 'BA S/E CODELCO VENTANAS 2 110 BP1', 'Voltaje'] = "110.0"
df['Voltaje'] = df['Voltaje'].str.replace(',', '.')
df['Voltaje'] = df['Voltaje'].astype(float)
df["Voltaje"] = np.trunc(df["Voltaje"]).astype(int)
df = df.dropna(subset=['Voltaje'])

In [36]:
df.head()

,Nombre,Nombre Subestación,Macrozona,Región,NS1,N1,Voltaje
0,BA S/E CENTRAL ALFALFAL 12KV BP1,S/E CENTRAL ALFALFAL,Centro,Metropolitana de Santiago,ALFALFAL,CENTRAL ALFALFAL,12
1,BA S/E CENTRAL ALFALFAL 12KV BP2,S/E CENTRAL ALFALFAL,Centro,Metropolitana de Santiago,ALFALFAL,CENTRAL ALFALFAL,12
2,BA S/E CENTRAL MAITENES 6.6KV B1,S/E CENTRAL MAITENES,Centro,Metropolitana de Santiago,MAITENES,CENTRAL MAITENES,6
3,BA S/E CENTRAL QUELTEHUES 110KV BP1,S/E CENTRAL QUELTEHUES,Centro,Metropolitana de Santiago,QUELTEHUES,CENTRAL QUELTEHUES,110
4,BA S/E CENTRAL QUELTEHUES 12KV,S/E CENTRAL QUELTEHUES,Centro,Metropolitana de Santiago,QUELTEHUES,CENTRAL QUELTEHUES,12


In [38]:
df['NS2'] = df['NS1'].astype(str) + "_" + df['Voltaje'].astype(str)
df.head()

,Nombre,Nombre Subestación,Macrozona,Región,NS1,N1,Voltaje,NS2
0,BA S/E CENTRAL ALFALFAL 12KV BP1,S/E CENTRAL ALFALFAL,Centro,Metropolitana de Santiago,ALFALFAL,CENTRAL ALFALFAL,12,ALFALFAL_12
1,BA S/E CENTRAL ALFALFAL 12KV BP2,S/E CENTRAL ALFALFAL,Centro,Metropolitana de Santiago,ALFALFAL,CENTRAL ALFALFAL,12,ALFALFAL_12
2,BA S/E CENTRAL MAITENES 6.6KV B1,S/E CENTRAL MAITENES,Centro,Metropolitana de Santiago,MAITENES,CENTRAL MAITENES,6,MAITENES_6
3,BA S/E CENTRAL QUELTEHUES 110KV BP1,S/E CENTRAL QUELTEHUES,Centro,Metropolitana de Santiago,QUELTEHUES,CENTRAL QUELTEHUES,110,QUELTEHUES_110
4,BA S/E CENTRAL QUELTEHUES 12KV,S/E CENTRAL QUELTEHUES,Centro,Metropolitana de Santiago,QUELTEHUES,CENTRAL QUELTEHUES,12,QUELTEHUES_12


In [ ]:
opciones_validas = df['NS2'].dropna().unique().tolist()

def obtener_top_5(nombre_buscar):
    if pd.isna(nombre_buscar):
        return pd.Series([None, 0] * 5)
        
    resultados = process.extract(
        str(nombre_buscar), 
        opciones_validas, 
        scorer=fuzz.token_sort_ratio,
        limit=5
    )
    
    fila_resultado = []
    for match in resultados:
        fila_resultado.extend([match[0], match[1]])
        
    while len(fila_resultado) < 10:
        fila_resultado.extend([None, 0])
        
    return pd.Series(fila_resultado)

dfu = dfp[['nombre_barra', 'tension']].dropna(subset=['nombre_barra']).drop_duplicates(subset=['nombre_barra']).copy()

# Opcional: reseteamos el índice para que la tabla quede limpia
dfu = dfu.reset_index(drop=True)

# 2. Pegamos la tensión directamente al nombre_barra original
dfu['nombre_compare'] = dfu['nombre_barra'].astype(str) + "_" + dfu['tension'].astype(str)

columnas_top = [
    'Match_1', 'Score_1', 
    'Match_2', 'Score_2', 
    'Match_3', 'Score_3', 
    'Match_4', 'Score_4', 
    'Match_5', 'Score_5'
]

dfu[columnas_top] = dfu['nombre_compare'].apply(obtener_top_5)


In [10]:
pd.reset_option('display.max_rows')
dfu

,nombre_barra,nombre_compare,Match_1,Score_1,Match_2,Score_2,Match_3,Score_3,Match_4,Score_4,Match_5,Score_5
0,DOMEYKO,DOMEYKO,DOMEYKO,100.000000,SVCDOMEYKO,82.352941,DONGOYO,57.142857,BOMBEO2,57.142857,BOMBEO3,57.142857
1,ALTONORTE,ALTONORTE,ALTONORTE,100.000000,ALGARROBONORTE,69.565217,PAMPANORTE,63.157895,ALTOBONITO,63.157895,LATORRE,62.500000
2,ANTOFAGASTA,ANTOFAGASTA,ANTOFAGASTA,100.000000,SANTAMARTA,66.666667,PLANTAMATTA,63.636364,SANTAROSA,60.000000,AGUASANTA,60.000000
3,ARICA,ARICA,ARICA,100.000000,MARISCAL,76.923077,LARUCA,72.727273,VILLARRICA,66.666667,LAREINA,66.666667
4,CHAPIQUINA,CHAPIQUINA,CHAPIQUIÑA,90.000000,MARIQUINA,73.684211,CHACAHUIN,63.157895,APOQUINDO,63.157895,CAUTIN,62.500000
...,...,...,...,...,...,...,...,...,...,...,...,...
488,LA_POLVORA,LAPOLVORA,LAPOLVORA,100.000000,LOMACOLORADA,66.666667,LAPORTADA,66.666667,ELSALVADOR,63.157895,LAPALMA,62.500000
489,SULFUROS,SULFUROS,SULFUROS,100.000000,LOSLOROS,62.500000,LOSQUILOS,58.823529,LOSLIRIOS,58.823529,LUCERO,57.142857
490,RECINTO,RECINTO,RECINTO,100.000000,RENGO,66.666667,EJERCITO,66.666667,CARRERAPINTO,63.157895,TRESPINOS,62.500000
491,TMELON,TMELON,ELMELON,76.923077,TUNELELMELON,66.666667,MEJILLONES,62.500000,ELPEÑON,61.538462,QUELLON,61.538462


In [11]:
tolerancia = 90

df_dudosos = dfu[dfu['Score_1'] < tolerancia].sort_values('Score_1')

print(f"Tienes {len(df_dudosos)} barras que requieren revisión manual.")
df_dudosos.to_excel('auditoria_cruces_norevisada.xlsx', index=False)

Tienes 136 barras que requieren revisión manual.


In [12]:
auditoria_path = Path(r"E:\ProyectoAnalisisElectrico\BarrasEstaciones\auditoria_cruces.xlsx")

In [13]:
dfa = pd.read_excel(auditoria_path)


In [14]:
dfa.head()

,nombre_barra,nombre_compare,Match_1
0,M.V.CEN.,M.V.CEN.,NaN
1,S.P.PAPELES,S.P.PAPELES,NaN
2,S.F.MOSTAZAL,S.F.MOSTAZAL,SANFRANCISCODEMOSTAZAL
3,R.TOMIC,R.TOMIC,RADOMIROTOMIC
4,NEPTUNO,NEPTUNO,NaN


In [ ]:
dfx = dfu.merge(
    dfa[['nombre_barra', 'Match_1']], 
    on='nombre_barra', 
    how='left', 
    suffixes=('', '_auditoria')
)

fue_auditada = dfx['nombre_barra'].isin(dfa['nombre_barra'])

dfx['Match_Definitivo'] = np.where(fue_auditada, dfx['Match_1_auditoria'], dfx['Match_1'])


,nombre_barra,Match_1,Match_1_auditoria,Match_Definitivo
0,DOMEYKO,DOMEYKO,NaN,DOMEYKO
1,ALTONORTE,ALTONORTE,NaN,ALTONORTE
2,ANTOFAGASTA,ANTOFAGASTA,NaN,ANTOFAGASTA
3,ARICA,ARICA,NaN,ARICA
4,CHAPIQUINA,CHAPIQUIÑA,NaN,CHAPIQUIÑA
5,ATACAMA,ATACAMA,NaN,ATACAMA
6,TBARRILES,T.OFFBARRILES,T.OFFBARRILES,T.OFFBARRILES
7,CALAMA,CALAMA,NaN,CALAMA
8,CD.ARICA,ARICA,ARICA,ARICA
9,MOLYCOP,MOLYCOP,NaN,MOLYCOP


In [17]:
df.columns

Index(['Nombre', 'Nombre Subestación', 'Macrozona', 'Región', 'NS1', 'N1',
       'Voltaje'],
      dtype='str')